# Experiment: 股票筛选中 “PE < 30 且 PB > 5” 的端到端复现

Objective:
- 通过真实的 `screening` router、Prisma 仓储和执行服务，复现“市盈率 < 30 且市净率 > 5”的筛选请求。
- 在当前环境下记录真实会话链路是否成功；若失败，保留精确错误，方便继续定位运行时问题。

Success criteria:
- 能创建策略和筛选会话，并输出会话状态、错误信息和命中数量。
- 如果可以拿到实时市场快照，则用同一套领域过滤和打分逻辑补跑一次，输出实际命中股票预览。


In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import textwrap
import urllib.request
from pathlib import Path
from pprint import pprint


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists():
            return candidate
    raise RuntimeError("Repository root not found")


ROOT = find_repo_root(Path.cwd().resolve())
NOTEBOOK_TEMP_DIR = ROOT / "temp" / "jupyter-notebook"
NOTEBOOK_TEMP_DIR.mkdir(parents=True, exist_ok=True)
TSX_CMD = ["npx", "tsx", "--env-file=.env", "-"]
PYTHON_SERVICE_HEALTH_URL = "http://localhost:8000/health"
PYTHON_SERVICE_PYTHON = ROOT / "python_services" / ".venv" / "Scripts" / "python.exe"
SNAPSHOT_PATH = NOTEBOOK_TEMP_DIR / "screening-pe-lt-30-pb-gt-5-universe.json"


def extract_result(stdout: str) -> dict:
    for line in reversed(stdout.splitlines()):
        if line.startswith("__RESULT__"):
            return json.loads(line[len("__RESULT__"):])
    raise ValueError("Missing __RESULT__ marker in output\n" + stdout)


def run_process(command, *, cwd=ROOT, input_text=None, extra_env=None, timeout=600):
    env = os.environ.copy()
    if extra_env:
        env.update({key: str(value) for key, value in extra_env.items()})

    completed = subprocess.run(
        command,
        cwd=str(cwd),
        input=input_text,
        text=True,
        capture_output=True,
        timeout=timeout,
        env=env,
        check=False,
    )
    return {
        "command": command,
        "cwd": str(cwd),
        "returncode": completed.returncode,
        "stdout": completed.stdout,
        "stderr": completed.stderr,
    }


def run_ts(script: str, *, timeout=600, extra_env=None):
    result = run_process(
        TSX_CMD,
        input_text=script,
        timeout=timeout,
        extra_env=extra_env,
    )
    parsed = None
    if "__RESULT__" in result["stdout"]:
        parsed = extract_result(result["stdout"])
    return {**result, "parsed": parsed}


def http_json(url: str, *, timeout=10):
    with urllib.request.urlopen(url, timeout=timeout) as response:
        return json.loads(response.read().decode("utf-8"))


NOTEBOOK_CONTEXT = {
    "root": str(ROOT),
    "temp_dir": str(NOTEBOOK_TEMP_DIR),
    "snapshot_path": str(SNAPSHOT_PATH),
    "python_service_health_url": PYTHON_SERVICE_HEALTH_URL,
    "python_service_python": str(PYTHON_SERVICE_PYTHON),
}
NOTEBOOK_CONTEXT


## Plan

- Hypothesis: 当前环境里的真实筛选链路可能会因为 Python 批量行情接口超时或上游数据不可达而失败。
- Variables to sweep: `PYTHON_SERVICE_TIMEOUT_MS`、本地市场快照是否能成功生成。
- Metrics to record: 会话状态、错误信息、`matchedCount`、`totalScanned`、功能性补跑的 `invalidCount` 与 `topPreview`。


In [ ]:
DB_PING_SCRIPT = textwrap.dedent(
    """
    import { db } from './src/server/db.ts';

    const summary = { mode: 'db_ping', success: false };
    try {
      const rows = await db.$queryRawUnsafe('SELECT 1 as ok');
      summary.success = true;
      summary.rows = rows;
    } catch (error) {
      summary.error = error instanceof Error ? error.message : String(error);
    } finally {
      console.log('__RESULT__' + JSON.stringify(summary));
      await db.$disconnect();
    }
    """
)

health_report = {
    "repo_root": str(ROOT),
    "python_service": None,
    "db": None,
}

try:
    health_report["python_service"] = {
        "success": True,
        "payload": http_json(PYTHON_SERVICE_HEALTH_URL, timeout=10),
    }
except Exception as error:  # noqa: BLE001
    health_report["python_service"] = {
        "success": False,
        "error": str(error),
    }

health_report["db"] = run_ts(
    DB_PING_SCRIPT,
    timeout=60,
    extra_env={"NODE_ENV": "production"},
)["parsed"]

pprint(health_report)


## Real chain

这一段直接走仓内真实入口：
- `screening` router 创建策略
- `executeStrategy` 建立会话
- `ScreeningExecutionService` 执行筛选

为了避免 dev 日志淹没输出，这里把 `NODE_ENV` 设为 `production`；同时把 Python 数据服务超时提升到 `300000ms`，尽量给全市场扫描留出更多时间。


In [ ]:
REAL_CHAIN_SCRIPT = textwrap.dedent(
    """
    import { db } from './src/server/db.ts';
    import { screeningRouter } from './src/server/api/routers/screening.ts';
    import { ScreeningExecutionService } from './src/server/application/screening/screening-execution-service.ts';
    import { ComparisonOperator } from './src/server/domain/screening/enums/comparison-operator.ts';
    import { IndicatorField } from './src/server/domain/screening/enums/indicator-field.ts';
    import { LogicalOperator } from './src/server/domain/screening/enums/logical-operator.ts';
    import { NormalizationMethod, ScoringDirection } from './src/server/domain/screening/value-objects/scoring-config.ts';
    import { PrismaScreeningSessionRepository } from './src/server/infrastructure/screening/prisma-screening-session-repository.ts';
    import { PrismaScreeningStrategyRepository } from './src/server/infrastructure/screening/prisma-screening-strategy-repository.ts';

    const NOTEBOOK_EMAIL = 'notebook-screening-e2e@local.test';
    const STRATEGY_NAME = '[Notebook] PE<30 and PB>5';
    const summary = {
      mode: 'real_screening_session',
      success: false,
      timeoutMs: process.env.PYTHON_SERVICE_TIMEOUT_MS ?? null,
    };

    try {
      const user = await db.user.upsert({
        where: { email: NOTEBOOK_EMAIL },
        update: { name: 'Notebook Screening E2E' },
        create: { email: NOTEBOOK_EMAIL, name: 'Notebook Screening E2E' },
      });

      await db.screeningSession.deleteMany({
        where: { userId: user.id, strategyName: { startsWith: STRATEGY_NAME } },
      });
      await db.screeningStrategy.deleteMany({
        where: { userId: user.id, name: { startsWith: STRATEGY_NAME } },
      });

      const caller = screeningRouter.createCaller({
        db,
        headers: new Headers(),
        session: {
          user: { id: user.id, name: user.name, email: user.email, image: user.image },
          expires: new Date(Date.now() + 3600_000).toISOString(),
        },
      });

      const strategy = await caller.createStrategy({
        name: STRATEGY_NAME,
        description: 'Notebook reproduction for PE < 30 and PB > 5',
        tags: ['notebook', 'e2e', 'screening'],
        isTemplate: false,
        filters: {
          groupId: crypto.randomUUID(),
          operator: LogicalOperator.AND,
          conditions: [
            {
              field: IndicatorField.PE,
              operator: ComparisonOperator.LESS_THAN,
              value: { type: 'numeric', value: 30 },
            },
            {
              field: IndicatorField.PB,
              operator: ComparisonOperator.GREATER_THAN,
              value: { type: 'numeric', value: 5 },
            },
          ],
          subGroups: [],
        },
        scoringConfig: {
          weights: {
            [IndicatorField.PE]: 0.5,
            [IndicatorField.PB]: 0.5,
          },
          directions: {
            [IndicatorField.PE]: ScoringDirection.DESC,
            [IndicatorField.PB]: ScoringDirection.ASC,
          },
          normalizationMethod: NormalizationMethod.MIN_MAX,
        },
      });

      const enqueueResult = await caller.executeStrategy({ strategyId: strategy.id });
      const sessionRepo = new PrismaScreeningSessionRepository(db);
      const strategyRepo = new PrismaScreeningStrategyRepository(db);
      const executionService = new ScreeningExecutionService({
        sessionRepository: sessionRepo,
        strategyRepository: strategyRepo,
      });
      const session = await sessionRepo.findById(enqueueResult.sessionId);
      if (!session) {
        throw new Error('Session not found after enqueue');
      }

      await (executionService).executeSession(session);
      const detail = await caller.getSessionDetail({ sessionId: enqueueResult.sessionId });

      Object.assign(summary, {
        success: detail.status === 'SUCCEEDED',
        userId: user.id,
        strategyId: strategy.id,
        sessionId: enqueueResult.sessionId,
        status: detail.status,
        currentStep: detail.currentStep,
        errorMessage: detail.errorMessage,
        matchedCount: detail.matchedCount,
        totalScanned: detail.totalScanned,
        executionTime: detail.executionTime,
      });
    } catch (error) {
      summary.fatalError = error instanceof Error ? error.message : String(error);
    } finally {
      console.log('__RESULT__' + JSON.stringify(summary));
      await db.$disconnect();
    }
    """
)

real_chain = run_ts(
    REAL_CHAIN_SCRIPT,
    timeout=900,
    extra_env={
        "NODE_ENV": "production",
        "PYTHON_SERVICE_TIMEOUT_MS": "300000",
    },
)["parsed"]

pprint(real_chain)


## Functional replay

如果真实 worker 链路在当前环境下失败，这一段会尝试做“功能性完整复现”：
- 先用本地 `python_services` 的 AkShare 适配器拉一份市场快照
- 再把快照交给 TypeScript 领域层，用同一套 `FilterGroup`、`FilterCondition`、`ScoringService` 复跑筛选

这样可以帮助我们区分“筛选规则本身有问题”和“运行时数据链路太慢或不可用”这两类故障。


In [ ]:
SNAPSHOT_SCRIPT = textwrap.dedent(
    f"""
    import json
    from pathlib import Path
    from app.services.akshare_adapter import AkShareAdapter

    result = {{
        'mode': 'snapshot_build',
        'success': False,
        'snapshotPath': r'{SNAPSHOT_PATH}',
    }}

    try:
        snapshot_path = Path(r'{SNAPSHOT_PATH}')
        snapshot_path.parent.mkdir(parents=True, exist_ok=True)
        universe = AkShareAdapter.get_stock_universe()
        snapshot_path.write_text(
            json.dumps(universe, ensure_ascii=False),
            encoding='utf-8',
        )
        result['success'] = True
        result['universeCount'] = len(universe)
    except Exception as error:  # noqa: BLE001
        result['error'] = str(error)

    print('__RESULT__' + json.dumps(result, ensure_ascii=False))
    """
)

if PYTHON_SERVICE_PYTHON.exists():
    snapshot_process = run_process(
        [str(PYTHON_SERVICE_PYTHON), '-'],
        cwd=ROOT / 'python_services',
        input_text=SNAPSHOT_SCRIPT,
        timeout=900,
    )
    snapshot_result = extract_result(snapshot_process['stdout'])
else:
    snapshot_result = {
        'mode': 'snapshot_build',
        'success': False,
        'error': f'Missing Python interpreter: {PYTHON_SERVICE_PYTHON}',
    }

FUNCTIONAL_REPLAY_SCRIPT = textwrap.dedent(
    """
    import { readFile } from 'node:fs/promises';
    import { FilterGroup } from './src/server/domain/screening/entities/filter-group.ts';
    import { Stock } from './src/server/domain/screening/entities/stock.ts';
    import { ComparisonOperator } from './src/server/domain/screening/enums/comparison-operator.ts';
    import { IndicatorField } from './src/server/domain/screening/enums/indicator-field.ts';
    import { LogicalOperator } from './src/server/domain/screening/enums/logical-operator.ts';
    import { IndicatorCalculationService } from './src/server/domain/screening/services/indicator-calculation-service.ts';
    import { ScoringService } from './src/server/domain/screening/services/scoring-service.ts';
    import { FilterCondition } from './src/server/domain/screening/value-objects/filter-condition.ts';
    import { NormalizationMethod, ScoringConfig, ScoringDirection } from './src/server/domain/screening/value-objects/scoring-config.ts';
    import { StockCode } from './src/server/domain/screening/value-objects/stock-code.ts';

    const summary = {
      mode: 'functional_replay',
      success: false,
      snapshotPath: process.env.NOTEBOOK_SNAPSHOT_PATH ?? null,
    };

    try {
      const raw = JSON.parse(
        await readFile(process.env.NOTEBOOK_SNAPSHOT_PATH ?? '', 'utf8')
      );
      const stocks = raw.map((item) =>
        new Stock({
          code: StockCode.create(String(item.code)),
          name: String(item.name ?? ''),
          industry: String(item.industry ?? '未知'),
          sector: String(item.sector ?? '主板'),
          roe: typeof item.roe === 'number' ? item.roe : null,
          pe: typeof item.pe === 'number' ? item.pe : null,
          pb: typeof item.pb === 'number' ? item.pb : null,
          eps: typeof item.eps === 'number' ? item.eps : null,
          revenue: typeof item.revenue === 'number' ? item.revenue : null,
          netProfit: typeof item.netProfit === 'number' ? item.netProfit : null,
          debtRatio: typeof item.debtRatio === 'number' ? item.debtRatio : null,
          marketCap: typeof item.marketCap === 'number' ? item.marketCap : null,
          floatMarketCap: typeof item.floatMarketCap === 'number' ? item.floatMarketCap : null,
          dataDate: item.dataDate ? new Date(String(item.dataDate)) : null,
        })
      );

      const filters = FilterGroup.create(
        LogicalOperator.AND,
        [
          FilterCondition.create(IndicatorField.PE, ComparisonOperator.LESS_THAN, {
            type: 'numeric',
            value: 30,
          }),
          FilterCondition.create(IndicatorField.PB, ComparisonOperator.GREATER_THAN, {
            type: 'numeric',
            value: 5,
          }),
        ],
        [],
      );

      const calcService = new IndicatorCalculationService({
        getIndicatorHistory: async () => [],
      });

      const matched = [];
      for (const stock of stocks) {
        if (await filters.matchAsync(stock, calcService)) {
          matched.push(stock);
        }
      }

      const scoringConfig = ScoringConfig.create(
        new Map([
          [IndicatorField.PE, 0.5],
          [IndicatorField.PB, 0.5],
        ]),
        NormalizationMethod.MIN_MAX,
        new Map([
          [IndicatorField.PE, ScoringDirection.DESC],
          [IndicatorField.PB, ScoringDirection.ASC],
        ]),
      );

      const scored = await new ScoringService().scoreStocks(
        matched,
        scoringConfig,
        calcService,
      );
      const invalidCount = matched.filter(
        (stock) =>
          !(
            typeof stock.pe === 'number' &&
            stock.pe < 30 &&
            typeof stock.pb === 'number' &&
            stock.pb > 5
          ),
      ).length;

      Object.assign(summary, {
        success: true,
        universeCount: stocks.length,
        matchedCount: matched.length,
        invalidCount,
        topPreview: scored.slice(0, 10).map((stock) => ({
          stockCode: stock.stockCode.value,
          stockName: stock.stockName,
          score: stock.score,
          pe: stock.indicatorValues.get(IndicatorField.PE),
          pb: stock.indicatorValues.get(IndicatorField.PB),
        })),
      });
    } catch (error) {
      summary.error = error instanceof Error ? error.message : String(error);
    }

    console.log('__RESULT__' + JSON.stringify(summary));
    """
)

functional_replay = None
if snapshot_result.get('success'):
    functional_replay = run_ts(
        FUNCTIONAL_REPLAY_SCRIPT,
        timeout=900,
        extra_env={
            'NODE_ENV': 'production',
            'NOTEBOOK_SNAPSHOT_PATH': str(SNAPSHOT_PATH),
        },
    )['parsed']
else:
    functional_replay = {
        'mode': 'functional_replay',
        'success': False,
        'error': 'Snapshot step did not succeed; see snapshot_result.error',
    }

pprint(snapshot_result)
pprint(functional_replay)


## Results

- 优先看 `real_chain`：它代表真实会话链路当前是否可运行。
- 如果 `real_chain.success` 为 `False`，再看 `snapshot_result` 和 `functional_replay`，判断是实时数据不可用，还是筛选逻辑本身出了问题。
- 如果 `functional_replay.success` 为 `True`，那么 `topPreview` 就是同一套筛选规则在当前快照上的实际命中结果。


In [ ]:
summary = {
    'health_report': health_report,
    'real_chain': real_chain,
    'snapshot_result': snapshot_result,
    'functional_replay': functional_replay,
}

pprint(summary)

if functional_replay and functional_replay.get('success'):
    assert functional_replay['invalidCount'] == 0, 'Replay output contains invalid matches'
    print('Functional replay produced only valid PE < 30 and PB > 5 matches.')
elif real_chain and real_chain.get('success'):
    print('Real screening session succeeded in the current environment.')
else:
    print('Current environment did not complete the full live run. Inspect real_chain.errorMessage, real_chain.fatalError, snapshot_result.error, and functional_replay.error for the blocker.')


## Next steps

- 如果 `real_chain.errorMessage` 指向 Python 数据服务超时，优先排查 `/api/v1/market/stocks/batch` 的批量性能，或者补一个更适合全市场扫描的一次性快照接口。
- 如果 `snapshot_result.error` 失败，优先检查 `python_services/.venv`、AkShare 出网能力、代理设置和上游站点可达性。
- 如果两条链路都成功，可以把 notebook 里的策略 payload 扩展成更多筛选组合，作为后续回归模板。
